# Initial Setup

In [ ]:
import io, math
from scipy.io import arff
from sklearn.metrics import max_error
from sklearn.metrics import mean_squared_error

import sklearn
import pandas as pd
import seaborn
import shap

# Data provisioning

The dataset consist of 28 columns. Ranging from 7 million to 22 million rows per column, with a total file size of 3.6gb. This immediately shows a processing problem where trying to do anything with the dataset requires alot of compute.

We can try and tackle this. By limiting the columns we actually need from the dataset. For now we gonna just take the colums that have been mentioned in the expert interview or are most closely related.

Also for these columns we dont need to load in all rows. Since the dataset contains a span of 20 years. So ideally we only want to take more recent data. For example the past 3 years.

While doing this we are gonna try and create a new filtered dataset. Consisting of fewer column and only past data rows of 3 years. That filtered dataset we will use further for our model.

# Data Dictionary
Identifiers  
Ticker: Company trading symbol  
Date: Date of the observation (price + ratios)  

Price / Market  
Adjusted Close (Target Variable): Realistic price including dividends and splits. Used for return calculations.  
Volume: Indicates market activity and sentiment around a stock.  
Market Cap: Company size. Smaller companies may have more pricing inefficiencies.  

Valuation (Core for value investing)  
P/E (ttm): Price relative to earnings. Used to assess how expensive a company is.  
P/S (ttm): Price relative to revenue. Useful when earnings are less reliable.  
P/B: Price relative to assets. Classic value investing metric.  
P/FCF (ttm): Price relative to free cash flow.  
EV/EBITDA: Measures value based on operational cash flow. Widely used by professionals.  
EV/Sales: Alternative valuation when profits are unstable.  

Value Signal  
Book to Market Value: Inverse of P/B. Indicates how “cheap” a stock is relative to its assets.  

Quality (avoid value traps).  
Operating Income/EV: Measures profitability relative to company value.  
Altman Z Score: Financial health indicator. Helps detect risk of bankruptcy.  


Optional  
Dividend Yield: Income return from dividends.  
Stakeholder mentioned dividend as an important anchor in valuation shows reliability of company.  

Data Choice  
The dataset contains both quarterly and yearly (TTM) data we will use past data of 3 years.  
For this model, TTM data is used because:

It reduces short-term noise  
It better reflects long-term earning capacity  
It aligns with how the stakeholder evaluates companies with long term vision in mind.  

In [ ]:
if False:
    df = pd.read_csv("us-derived-shareprices-daily/us-derived-shareprices-daily.csv")
    df.columns = df.columns.str.strip()

    print(df.columns.tolist())

['Ticker;SimFinId;Date;Open;High;Low;Close;Adj. Close;Volume;Dividend;Shares Outstanding;Market-Cap;Price to Earnings Ratio (quarterly);Price to Earnings Ratio (ttm);Price to Sales Ratio (quarterly);Price to Sales Ratio (ttm);Price to Book Value;Price to Free Cash Flow (quarterly);Price to Free Cash Flow (ttm);Enterprise Value;EV/EBITDA;EV/Sales;EV/FCF;Book to Market Value;Operating Income/EV;Altman Z Score;Dividend Yield;Price to Earnings Ratio (Adjusted)']


Make sure we select the right columns.

In [ ]:
# Data filtering step (executed once to create smaller working dataset)
if False:

    # columns we want to use
    cols = [
        "Ticker", 
        "Date", 
        "Adj. Close", 
        "Volume", 
        "Market-Cap",
        "Price to Earnings Ratio (ttm)",
        "Price to Sales Ratio (ttm)",
        "Price to Book Value",
        "Price to Free Cash Flow (ttm)",
        "EV/EBITDA", 
        "EV/Sales",
        "Book to Market Value",
        "Operating Income/EV",
        "Altman Z Score",
        "Dividend Yield"
    ]

    # load only selected columns
    df = pd.read_csv(
        "us-derived-shareprices-daily/us-derived-shareprices-daily.csv",
        usecols=cols,
        sep=";"
    )

    # convert Date column
    df["Date"] = pd.to_datetime(df["Date"])

    # filter between 2022 and end of 2025
    df = df[df["Date"].between("2022-01-01", "2025-12-31")]

    # save filtered dataset
    df.to_csv("filtered_shares_ratios_3years.csv", index=False)

    print("created filtered dataset")

created filtered dataset


Created a filtered dataset reduced from 28 to 15 columns

Sample

ToDo:

data describe method gebruiken
gaten in data duidelijk in kaart brengen
duidelijk maken wat moet mee gebeuren, en waarom onderbouwing!
ene attribuut 50% andere 75%
welke attributen gebruiken en waarom


In [15]:
df.sample(10)

,Ticker,Date,Adj. Close,Volume,Market-Cap,Price to Earnings Ratio (ttm),Price to Sales Ratio (ttm),Price to Book Value,Price to Free Cash Flow (ttm),EV/EBITDA,EV/Sales,Book to Market Value,Operating Income/EV,Altman Z Score,Dividend Yield
2672860,BIIB,2023-04-25,282.41,1527989,4.078000e+10,13.44852,4.16714,3.05413,10.04948,11.61831,4.15769,0.32743,0.07446,4.38207,NaN
7469461,FCPT,2022-04-12,22.82,1265868,2.163992e+09,25.28590,10.85372,2.24506,-19.46614,19.96714,15.23489,0.44542,0.03862,1.67547,0.04478
14747479,OHI,2025-02-18,33.77,2253452,1.016304e+10,22.90706,8.85279,1.96746,20.08120,15.60866,13.96031,0.50827,0.04331,2.27941,0.07364
3360965,CABA,2024-10-24,3.85,1037350,1.880674e+08,-0.48791,NaN,0.22005,NaN,0.15371,NaN,4.54442,6.60553,NaN,NaN
14035390,NLTX,2023-08-28,13.78,6112,3.033076e+07,-0.79258,NaN,0.48079,NaN,1.10165,NaN,2.07991,0.98414,NaN,NaN
3668596,CCB,2025-03-31,90.41,157041,1.356683e+09,27.00773,2.10116,2.78380,NaN,NaN,2.34807,0.35922,0.04200,NaN,NaN
8149750,FSTR,2022-07-15,13.83,15591,1.481884e+08,44.38480,0.29671,0.81052,43.44211,10.69866,0.35892,1.23378,0.01597,3.00185,NaN
7428421,FCCO,2022-12-07,18.69,2846,1.536324e+08,NaN,NaN,1.34393,NaN,NaN,NaN,0.74409,NaN,NaN,0.02499
15078496,OTLY,2023-03-16,43.40,214634,1.285334e+09,NaN,1.78094,1.62109,NaN,-5.15149,1.38237,NaN,-0.24089,2.12031,NaN
6169389,DSP,2022-05-02,6.00,194492,8.285400e+07,-9.58450,0.33099,0.26253,NaN,-2.66281,0.37606,3.80909,-0.50772,1.43707,NaN
